# RLSF omega selection: the dev-slice best-of-N pool

---
## 1 — Setup

In [1]:
# 7B bf16 is ~15 GB of weights and the COMET encoder shares the card: 24 GB is comfortable.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4090, 24564 MiB


In [2]:
from pathlib import Path

# %cd into a repo that is already the working directory clones a second copy underneath it,
# so the guard is manage.py, not the directory name.
if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

Cloning into 'Style-Aware-MT'...
remote: Enumerating objects: 1454, done.
remote: Counting objects: 100% (1454/1454), done.
remote: Compressing objects: 100% (652/652), done.
remote: Total 1454 (delta 978), reused 1266 (delta 790), pack-reused 0 (from 0)
Receiving objects: 100% (1454/1454), 15.93 MiB | 18.27 MiB/s, done.
Resolving deltas: 100% (978/978), done.
/workspace/Style-Aware-MT/notebooks/Style-Aware-MT
Already up to date.
29c540a


In [3]:
import subprocess
import sys

# The one interpreter every shell cell below runs through. `python3` on a rented host is a
# different install from the kernel, and the two stacks drift the moment either is upgraded.
PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
print('kernel', PY)

kernel /venv/main/bin/python


In [4]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [5]:
import numpy
import transformers

# COMET gets its own interpreter. Installing requirements-comet.txt into the kernel downgrades
# transformers and numpy under the generator, which then runs on a stack nothing else uses.
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

print(f'kernel transformers {transformers.__version__}, numpy {numpy.__version__}')
assert transformers.__version__.startswith('5.'), "COMET's pins landed in the kernel"
assert numpy.__version__.startswith('2.'), "COMET's pins landed in the kernel"

/workspace/Style-Aware-MT/notebooks/Style-Aware-MT/.venv-comet/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


comet ok
kernel transformers 5.12.1, numpy 2.4.1


In [6]:
import getpass
import logging
import os

# Set here rather than in a shell cell: the CLI runs and the Kiwi worker are children of this
# kernel and inherit os.environ, so this is the one place the keys have to exist.
for var in ('OPENAI_API_KEY', 'HF_TOKEN'):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
logging.getLogger('httpx').setLevel(logging.WARNING)
print({var: bool(os.environ.get(var)) for var in ('OPENAI_API_KEY', 'HF_TOKEN')})

OPENAI_API_KEY:  ········
HF_TOKEN:  ········


{'OPENAI_API_KEY': True, 'HF_TOKEN': True}


In [7]:
# wmt22-cometkiwi-da is gated, and the worker is what needs the access; failing here beats
# failing an hour into the sampling.
check = '''
import os
from huggingface_hub import HfApi, __version__
api, tok = HfApi(), os.environ.get("HF_TOKEN")
print("hub", __version__, "| whoami:", api.whoami(token=tok)["name"])
api.list_repo_files("Unbabel/wmt22-cometkiwi-da", token=tok)
print("cometkiwi access OK")
'''
subprocess.run([COMET_PY, '-c', check], check=True)

hub 0.36.2 | whoami: prnamhr
cometkiwi access OK


CompletedProcess(args=['.venv-comet/bin/python', '-c', '\nimport os\nfrom huggingface_hub import HfApi, __version__\napi, tok = HfApi(), os.environ.get("HF_TOKEN")\nprint("hub", __version__, "| whoami:", api.whoami(token=tok)["name"])\napi.list_repo_files("Unbabel/wmt22-cometkiwi-da", token=tok)\nprint("cometkiwi access OK")\n'], returncode=0)

---
## 2 — Pre-flight

In [10]:
import hashlib
import json
import math

import yaml

from src.rlsf.config import judge_concurrency, load_config, make_judge_client, reward_config
from src.rlsf.pool import measured_per_call_usd, plan, pool_settings, read_dev
from src.rlsf.reward import load_train_template

CONFIG = 'configs/rlsf.yaml'

# The caps were declared 2026-08-08, so this loads with require_caps on.
cfg = load_config(CONFIG)
settings = pool_settings(cfg)
N = settings['n']
G = cfg['rlsf']['rollout']['group_size']
sources, refs = read_dev(cfg['data']['dev_file'])

# -- the pool's own ceiling, separate from the training caps: this notebook may spend
#    pool.judge_calls and no more, whatever caps.max_judge_calls allows
assert len(sources) * N == settings['judge_calls'], (
    f"{len(sources)} segments x N={N} is {len(sources) * N} calls against a declared "
    f"ceiling of {settings['judge_calls']}"
)
assert settings['judge_calls'] <= cfg['rlsf']['caps']['max_judge_calls']
assert N <= cfg['rlsf']['caps']['group_size_ceiling']

# -- the locked control: a quantized or swapped base is a different experiment
gen = cfg['generator']
assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
assert gen['load_in_4bit'] is False, 'quantizing redefines the frozen base'
assert Path(gen['adapter_path'], 'adapter_config.json').exists(), (
    f"{gen['adapter_path']} is missing; copy the frozen PEFT checkpoint onto this host first"
)

# -- greedy sampling gives a group zero variance and the whole pool is uninformative
assert cfg['rlsf']['rollout']['temperature'] > 0, 'greedy samples cannot be ranked'

# -- reward_config rescales to unit ||omega||, so these are not the raw config numbers
w = reward_config(cfg).weights
print(f"policy {gen['model']} + {gen['adapter_path']}")
print(f"pool   {len(sources)} dev segments x N={N} at T={cfg['rlsf']['rollout']['temperature']}")
print('reward', {k: round(v, 3) for k, v in w.items()},
      f"||omega|| = {math.hypot(*w.values()):.3f}")

policy Qwen/Qwen2.5-7B-Instruct + models/peft_lora_r32_lr2e-4/checkpoint-1358
pool   499 dev segments x N=8 at T=1.0
reward {'bleu': 0.577, 'kiwi': 0.577, 'judge': 0.577} ||omega|| = 1.000


In [11]:
# -- the reward judge must not be either evaluation rater, or the pool spends a rater on
#    the one condition that most needs a rater it was not trained against
raters = {yaml.safe_load(Path(p).read_text())['judge']['model']
          for p in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml')}
assert cfg['judge']['model'] not in raters, (cfg['judge']['model'], raters)

# -- seeded, because under group normalization a rater flipping a 3 to a 4 inverts an
#    advantage sign, and this pool is re-ranked four times off one set of verdicts
assert cfg['judge']['temperature'] == 0.0 and cfg['judge']['seed'] == 42

# -- the rubric must be the frozen one; load_train_template raises on drift, this reports it
text = load_train_template()
digest = hashlib.sha256(text.encode()).hexdigest()
frozen = json.loads(Path('prompts/hashes.json').read_text())['templates']
assert digest == frozen['judge_train.txt']['sha256']
assert cfg['template_file'] == 'prompts/judge_train.txt', 'the eval rubric would be circular'

print(f"reward judge {cfg['judge']['model']}, distinct from {sorted(raters)}")
print(f"rubric verified {digest[:16]}")

reward judge gpt-4o-mini, distinct from ['claude-haiku-4-5', 'gpt-5.6-terra']
rubric verified 8eaa11ff341c86ca


In [12]:
# -- the dev slice, against the manifest written when it was carved
man = json.loads(Path('data/splits/rlsf_dev_manifest.json').read_text())
for name, want in man['hashes'].items():
    got = hashlib.sha256((Path('data/splits') / name).read_bytes()).hexdigest()
    assert got == want, f'{name} differs from the manifest'
print(f"dev slice {man['counts']['rlsf_dev']} segments, {man['counts']['dev_works']} works")

# -- the slice is not unseen by the model; it selects weights, it does not measure them
print('\n'.join('  ' + c for c in man['caveats']))

dev slice 499 segments, 4 works
  Held out from PPO updates only. The PEFT checkpoint RLSF initializes from was trained on all of train.jsonl, this slice included, so the slice is not unseen by the model.
  results/stylometrics_centroid.json was built over all 10,860 train targets, including these works.
  Dev-slice figures select the reward weights and are never reported as a result; val remains the reported split.


In [13]:
# The rate is measured, not assumed: token counts from the smoke priced at whatever the
# client charges for this model today. docs/budget.md quotes $0.29 for the whole pool.
judge_client = make_judge_client(cfg)
rate = measured_per_call_usd(judge_client, cfg['judge']['model'])
p = plan(len(sources), N, rate)

print(f"{p['samples']} completions, {p['judge_calls']} judge calls at concurrency "
      f"{judge_concurrency(cfg)}")
print(f"${rate:.3e} per call -> ${p['est_usd']:.2f}  (docs/budget.md: $0.29)")
print(f"pool ceiling {settings['judge_calls']} calls; "
      f"caps {cfg['rlsf']['caps']['max_judge_calls']} calls / "
      f"${cfg['rlsf']['caps']['max_judge_spend_usd']}")

3992 completions, 3992 judge calls at concurrency 8
$7.366e-05 per call -> $0.29  (docs/budget.md: $0.29)
pool ceiling 3992 calls; caps 340000 calls / $25.0


---
## 3 — Free pass

In [12]:
!{PY} manage.py rlsf_pool --config {CONFIG} --segments 2 --skip_judge \
    --out outputs/rlsf/pool_free.jsonl

plan: 2 segments x N=8 = 16 samples
      judge skipped: 0 paid calls, judge component held flat
config.json: 100%|████████████████████████████| 663/663 [00:00<00:00, 3.23MB/s]
tokenizer_config.json: 100%|██████████████| 7.30k/7.30k [00:00<00:00, 5.65MB/s]
vocab.json: 100%|█████████████████████████| 2.78M/2.78M [00:00<00:00, 3.82MB/s]
merges.txt: 100%|█████████████████████████| 1.67M/1.67M [00:00<00:00, 24.6MB/s]
tokenizer.json: 100%|█████████████████████| 7.03M/7.03M [00:00<00:00, 37.5MB/s]
model.safetensors.index.json: 100%|███████| 27.8k/27.8k [00:00<00:00, 33.2MB/s]
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 4 files:   0%|                                  | 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 3.86GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 7.73GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 15.2GB            
Reconstructing (incomp

---
## 4 — Paid pass

In [13]:
!{PY} manage.py rlsf_pool --config {CONFIG} --resume --yes

plan: 499 segments x N=8 = 3992 samples
      3992 judge calls at concurrency 8, ~$0.2941 at a measured $7.366e-05/call (pool ceiling 3992)
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 134.04it/s]
policy Qwen/Qwen2.5-7B-Instruct + models/peft_lora_r32_lr2e-4/checkpoint-1358
kiwi worker ready (reference_free=True)
  25/499 segments, 1.7 min elapsed, 31.7 min left, 200 judge calls ($0.0146)
  50/499 segments, 2.9 min elapsed, 25.6 min left, 400 judge calls ($0.0294)
  75/499 segments, 4.1 min elapsed, 23.1 min left, 600 judge calls ($0.0443)
  100/499 segments, 5.3 min elapsed, 21.2 min left, 800 judge calls ($0.0591)
  125/499 segments, 6.7 min elapsed, 20.1 min left, 1000 judge calls ($0.0747)
  150/499 segments, 7.9 min elapsed, 18.5 min left, 1200 judge calls ($0.0898)
  175/499 segments, 9.2 min elapsed, 17.0 min left, 1400 judge calls ($0.1048)
  200/499 segments, 10.5 min elapsed, 15.8 min left, 1600 judge calls ($0.1202)
  225/499 segments, 11.9 min elapsed

---
## 5 — Read the pool

In [14]:
from src.rlsf.pool import read_pool, sidecar

POOL = Path(cfg['output']['pool'])
rows = read_pool(POOL)
sizes = {len(r['hyps']) for r in rows}
unmeasured = sum(v is None or v != v for r in rows for v in r['scores']['judge'])

print(f"{POOL}: {len(rows)} segments, {sorted(sizes)} completions each, "
      f"{POOL.stat().st_size / 1e6:.1f} MB")
assert sizes == {N} and len(rows) == len(sources), 'the pool is incomplete'
print(f"{unmeasured} samples have no judge verdict "
      f"({unmeasured / (len(rows) * N):.2%}); those are dropped, not scored 1")

outputs/rlsf/pool.jsonl: 499 segments, [8] completions each, 0.9 MB
0 samples have no judge verdict (0.00%); those are dropped, not scored 1


In [15]:
# The measured rate this pool paid, against what section 2 planned and what budget.md holds.
u = json.loads(sidecar(POOL, 'usage.json').read_text())
print(f"{u['calls']} calls, {u['prompt_tokens'] / u['calls']:.0f} in / "
      f"{u['completion_tokens'] / u['calls']:.0f} out per call")
print(f"${u['cost_usd']:.4f} total, ${u['per_call_usd']:.6f}/call  (planned ${p['est_usd']:.2f})")
# The sidecar records what the fan-out achieved, not what it asked for; that is the config's.
print(f"judge block {u['wall_s'] / 60:.1f} min wall at concurrency {judge_concurrency(cfg)}, "
      f"{u['achieved_parallelism']:.1f}x achieved")

3992 calls, 430 in / 18 out per call
$0.3013 total, $0.000075/call  (planned $0.29)
judge block 5.8 min wall at concurrency 8, 7.8x achieved


---
## 6 — Commit the pool, then release the GPU

In [16]:
for path in (POOL, sidecar(POOL, 'manifest.json'), sidecar(POOL, 'usage.json')):
    print(f"{path.stat().st_size / 1e6:8.2f} MB  {path}")
print()
print(json.dumps(json.loads(sidecar(POOL, 'manifest.json').read_text())['hashes'], indent=2))

    0.89 MB  outputs/rlsf/pool.jsonl
    0.00 MB  outputs/rlsf/pool_manifest.json
    0.00 MB  outputs/rlsf/pool_usage.json

{
  "rlsf_dev.jsonl": "45b45e40b03ea3c3a49cc7b374304870aad0c023e5e47cbacb47a9cc72b4f8e9",
  "pool.jsonl": "20935cdb729800e4261b7027cedb59409b485a45ab6a1bd2b06f1f1d11e4e6fd"
}


---
## 7 — Rank the grid

In [18]:
!{PY} manage.py rlsf_omega --config {CONFIG}

outputs/rlsf/pool.jsonl: 499 segments x N=8 = 3992 samples

per-component degeneracy at N=8: what each term can separate on its own
  bleu     4/499 flat groups (1%)
  kiwi     4/499 flat groups (1%)
  judge    19/499 flat groups (4%)

cell          deg@8   deg@4  picks    bleu   judge    kiwi      stylo marker_rate dz
  w3_0.0         1%      2%    497  46.543   3.398   0.701      0.567   +0.022 +/- 0.024
  w3_0.5         1%      1%    497  46.773   3.712   0.697      0.616   +0.082 +/- 0.026
  w3_1.0         1%      1%    497  44.862   3.952   0.696      0.673   +0.134 +/- 0.026
  w3_2.0         1%      1%    497  41.632   4.197   0.688      0.727   +0.195 +/- 0.026
  random                       497  30.067   3.235   0.662      0.566   +0.016 +/- 0.026
  pool                        3992  29.116   3.252   0.665      0.553   +0.000 +/- 0.000

Degeneracy at N=8 and at G=4 are not comparable: a larger group degenerates less by construction, because more draws is more chances to differ. 

---
## 8 — The three readings

In [23]:
sel = json.loads(sidecar(POOL, 'omega.json').read_text())
feature = sel['feature']

for name, s in sel['per_component_degeneracy'].items():
    print(f"   {name:8s} flat in {s['degenerate']}/{s['groups']} groups "
          f"({s['degenerate_frac']:.1%}) at N={sel['n']}")
print()
for c in sel['cells']:
    sub = c['subgroup']
    print(f"   {c['cell']:8s} combined reward flat in {c['degenerate_frac']:.1%} of groups "
          f"at N={c['n']}, {sub['degenerate_frac']:.1%} at G={sub['group_size']}")

   bleu     flat in 4/499 groups (0.8%) at N=8
   kiwi     flat in 4/499 groups (0.8%) at N=8
   judge    flat in 19/499 groups (3.8%) at N=8

   w3_0.0   combined reward flat in 1.0% of groups at N=8, 2.4% at G=4
   w3_0.5   combined reward flat in 0.8% of groups at N=8, 1.4% at G=4
   w3_1.0   combined reward flat in 0.8% of groups at N=8, 1.4% at G=4
   w3_2.0   combined reward flat in 0.8% of groups at N=8, 1.4% at G=4


In [24]:
v = sel['selection']
if v['cell']:
    print(f"   {v['cell']}: {v['reason']}")
    print(f"   set rlsf.reward to {v['weights']}")
else:
    print(f"   none: {v['reason']}")
for name, why in v['rejected'].items():
    print(f"   rejected {name}: {why}")

   w3_0.0: lowest register distance (0.567) of the 4 cells that give most groups a gradient
   set rlsf.reward to {'name': 'w3_0.0', 'w_kiwi': 1.0, 'w_bleu': 1.0, 'w_judge': 0.0}


In [26]:
print(f"   {'cell':10s} {'stylo_dist':>10s} {feature + ' dz':>16s}")
for c in sel['cells'] + [{'cell': k, 'picks': a} for k, a in sel['anchors'].items()]:
    s = c['picks'][f'{feature}_shift']
    print(f"   {c['cell']:10s} {c['picks']['stylo_dist']:10.3f} "
          f"{s['delta']:+9.3f} +/- {s['se']:.3f}")

g = sel['selection'].get('goodhart')
if g:
    print(f"\n   The selected cell's picks sit {g['delta']:+.2f} +/- {g['se']:.2f} above their "
          f"own groups,\n   against the {g['threshold']:.2f} band the drift rule halts a run "
          f"for.")
    if g['over_threshold']:
        print("   Best-of-N is the ceiling on what this reward can pull the policy toward, "
              "and it\n   already clears the band. Expect GRPO to trip the drift stop.")
    else:
        print("   Best-of-N is the ceiling on what this reward can pull the policy toward, "
              "and it\n   stays inside the band.")

   cell       stylo_dist   marker_rate dz
   w3_0.0          0.567    +0.022 +/- 0.024
   w3_0.5          0.616    +0.082 +/- 0.026
   w3_1.0          0.673    +0.134 +/- 0.026
   w3_2.0          0.727    +0.195 +/- 0.026
   random          0.566    +0.016 +/- 0.026
   pool            0.553    +0.000 +/- 0.000

   The selected cell's picks sit +0.02 +/- 0.02 above their own groups,
   against the 0.23 band the drift rule halts a run for.
   Best-of-N is the ceiling on what this reward can pull the policy toward, and it
   stays inside the band.
